In [ ]:
pip install transformers datasets peft accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


In [ ]:
pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2


In [ ]:
import os
import random
from typing import Dict, Any, Callable, Tuple, List
import math
import time
import json

import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model, TaskType


from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))


In [ ]:
## set seed
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

#  1. Data loading & preprocessing

def load_emotion_dataset(
    model_name: str = "distilbert-base-uncased",
    max_length: int = 128,
    train_subset: int = 3000,
    batch_size: int = 32,
):

    dataset = load_dataset("dair-ai/emotion")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        enc["labels"] = batch["label"]
        return enc

    encoded = dataset.map(preprocess, batched=True)

    if train_subset is not None and train_subset < len(encoded["train"]):
        encoded["train"] = encoded["train"].shuffle(seed=42).select(range(train_subset))

    encoded.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"],
    )

    train_loader = DataLoader(encoded["train"], batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(encoded["validation"], batch_size=batch_size)
    test_loader = DataLoader(encoded["test"], batch_size=batch_size)

    num_labels = dataset["train"].features["label"].num_classes
    id2label = {i: dataset["train"].features["label"].int2str(i) for i in range(num_labels)}
    label2id = {v: k for k, v in id2label.items()}

    return train_loader, val_loader, test_loader, tokenizer, num_labels, id2label, label2id

#  2. LoRA model construction and training

def build_lora_model(
    base_model_name: str,
    num_labels: int,
    cfg: Dict[str, Any],
    id2label: Dict[int, str],
    label2id: Dict[str, int],
):
    """
    Build a DistilBERT + LoRA classification model according to cfg:
        cfg["lora_r"], cfg["lora_alpha"], cfg["lora_dropout"], cfg["target_modules"]
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )

    # Freeze base model to train only LoRA + classifier
    for param in model.base_model.parameters():
        param.requires_grad = False

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["target_modules"],
    )

    model = get_peft_model(model, lora_config)
    return model


def train_one_model(
    cfg: Dict[str, Any],
    train_loader: DataLoader,
    val_loader: DataLoader,
    base_model_name: str,
    num_labels: int,
    id2label: Dict[int, str],
    label2id: Dict[str, int],
    device: torch.device,
    num_epochs: int = 3,
) -> float:
    """
    Train DistilBERT + LoRA with hyperparameters in cfg.
    Returns validation accuracy (float in [0,1]).
    """
    model = build_lora_model(
        base_model_name=base_model_name,
        num_labels=num_labels,
        cfg=cfg,
        id2label=id2label,
        label2id=label2id,
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=cfg["learning_rate"])

    total_steps = num_epochs * len(train_loader)
    warmup_steps = int(cfg["warmup_ratio"] * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"    Epoch {epoch+1}/{num_epochs} - train loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            labels = batch["labels"]
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total if total > 0 else 0.0

    del model
    torch.cuda.empty_cache()

    return val_acc


In [ ]:
#  3. Hyperparameter decoding for AO

def decode_candidate(x: np.ndarray) -> Dict[str, Any]:
    rank_choices = [2, 4, 8, 16, 32]
    alpha_choices = [8, 16, 32, 64, 128]

    def pick_choice(val: float, choices):
        idx = int(round(val * (len(choices) - 1)))
        idx = max(0, min(len(choices) - 1, idx))
        return choices[idx]

    lr_min = 1e-6
    lr_max = 5e-4
    log_min = math.log(lr_min)
    log_max = math.log(lr_max)
    log_lr = log_min + float(x[0]) * (log_max - log_min)
    learning_rate = math.exp(log_lr)

    warmup_ratio = 0.2 * float(x[1])

    lora_r = pick_choice(x[2], rank_choices)
    lora_alpha = pick_choice(x[3], alpha_choices)

    # LoRA dropout: [0.0, 0.3]
    lora_dropout = 0.3 * float(x[4])

    # Target modules: categorical
    if x[5] < 0.5:
        target_modules = ["q_lin", "v_lin"]  # attention-only
    else:
        target_modules = ["q_lin", "v_lin", "ffn.lin1", "ffn.lin2"]  # attention+ffn

    cfg = {
        "learning_rate": learning_rate,
        "warmup_ratio": warmup_ratio,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "lora_dropout": lora_dropout,
        "target_modules": target_modules,
    }
    return cfg


In [ ]:
#  4. Levy flight helper

def levy_flight(beta: float = 1.5, size: int = 1) -> np.ndarray:
    """
    Generate Levy distribution random vector.
    """
    sigma = (
        math.gamma(1 + beta)
        * np.sin(np.pi * beta / 2)
        / (math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2))
    ) ** (1 / beta)
    u = np.random.normal(0, sigma, size)
    v = np.random.normal(0, 1, size)
    step = u / (np.abs(v) ** (1 / beta))
    return step



#  5. Aquila Optimizer (AO) with logging (per seed)

class AquilaOptimizer:
    def __init__(
        self,
        dim: int,
        fitness_function: Callable[[Dict[str, Any]], float],
        device: torch.device,
        pop_size: int = 10,
        iterations: int = 10,
    ):
        """
        dim: dimension of continuous search space (here 6)
        fitness_function: takes decoded cfg dict, returns scalar to MAXIMISE
                          (validation accuracy).
        """
        self.dim = dim
        self.fitness = fitness_function
        self.N = pop_size
        self.T = iterations
        self.device = device

        self.X = np.random.rand(self.N, dim)
        self.F = np.zeros(self.N)

        self.eval_counter: int = 0
        self.trials: List[Dict[str, Any]] = []

        self.candidate_best_val_acc = np.full(self.N, -np.inf, dtype=float)
        self.candidate_best_positions = [self.X[i].copy() for i in range(self.N)]

        self.global_best_val_acc: float = float("-inf")
        self.global_best_trial: int = 0
        self.global_best_candidate_no: int = -1
        self.global_best_position: np.ndarray = self.X[0].copy()


        self.last_train_time_sec: float = 0.0
        self.last_gpu_memory_gb: float = 0.0

    def enforce_bounds(self, X: np.ndarray) -> np.ndarray:
        return np.clip(X, 0.0, 1.0)

    def _make_candidate_log(
        self,
        candidate_no: int,
        phase: str,
        step_size: float,
        cfg: Dict[str, Any],
        val_acc: float,
        candidate_best_val_acc: float,
        candidate_best_position: np.ndarray,
        TF: float,
        r2: float,
        levy_lambda: float = 1.5,
    ) -> Dict[str, Any]:

        if len(cfg["target_modules"]) == 2:
            tm_cat = "attention-only"
        else:
            tm_cat = "attention-plus-ffn"

        return {
            "candidate_no": candidate_no,
            "gpu_memory_gb": float(self.last_gpu_memory_gb),
            "val_acc": float(val_acc),
            "train_time_sec": float(self.last_train_time_sec),
            "hyperparams": {
                "learning_rate": float(cfg["learning_rate"]),
                "warmup_ratio": float(cfg["warmup_ratio"]),
                "lora_r": int(cfg["lora_r"]),
                "lora_alpha": int(cfg["lora_alpha"]),
                "lora_dropout": float(cfg["lora_dropout"]),
                "target_modules": tm_cat,
            },
            "ao_state": {
                "phase": phase,
                "search_step_size": float(step_size),
                "current_candidate_best_val_acc": float(candidate_best_val_acc),
                "best_position_so_far": candidate_best_position.tolist(),
                "control_params": {
                    "alpha": float(TF),
                    "beta": float(r2),
                    "levy_lambda": float(levy_lambda),
                },
            },
        }

    def _update_timing_and_gpu_memory(self, start_t: float, end_t: float):
        self.last_train_time_sec = end_t - start_t
        if torch.cuda.is_available() and self.device.type == "cuda":
            mem_bytes = torch.cuda.max_memory_allocated(self.device)
            self.last_gpu_memory_gb = mem_bytes / (1024 ** 3)
            torch.cuda.reset_peak_memory_stats(self.device)
        else:
            self.last_gpu_memory_gb = 0.0

    def optimize(self) -> Tuple[np.ndarray, float, Dict[str, Any]]:
        f_best = float("-inf")
        X_best = self.X[0].copy()

        for i in range(self.N):
            cfg = decode_candidate(self.X[i])

            if torch.cuda.is_available() and self.device.type == "cuda":
                torch.cuda.reset_peak_memory_stats(self.device)

            start_t = time.perf_counter()
            val_acc = self.fitness(cfg)
            end_t = time.perf_counter()

            self._update_timing_and_gpu_memory(start_t, end_t)
            self.eval_counter += 1
            self.F[i] = val_acc

            # per-candidate personal best
            if val_acc > self.candidate_best_val_acc[i]:
                self.candidate_best_val_acc[i] = val_acc
                self.candidate_best_positions[i] = self.X[i].copy()

            # global best over all candidates
            if val_acc > f_best:
                f_best = val_acc
                X_best = self.X[i].copy()

            if val_acc > self.global_best_val_acc:
                self.global_best_val_acc = val_acc
                self.global_best_trial = 0  # initial phase
                self.global_best_candidate_no = self.eval_counter
                self.global_best_position = X_best.copy()

        print(f"[AO] Initial best fitness (val_acc) = {f_best:.6f}")

        # Main AO loop (trials = iterations)
        for t in range(1, self.T + 1):
            X_M = np.mean(self.X, axis=0)
            TF = t / self.T

            trial_candidates: List[Dict[str, Any]] = []

            for i in range(self.N):
                r1, r2 = random.random(), random.random()
                phase = "exploration" if TF < 2 / 3 else "exploitation"

                old_X = self.X[i].copy()

                if TF < 2 / 3:  # Exploration
                    if r1 < 0.5:
                        # X1 – expanded exploration
                        X_new = X_best * (1 - TF) + (X_M - X_best * r2)
                    else:
                        # X2 – narrowed exploration (Levy)
                        levy_step = levy_flight(size=self.dim)
                        XR = self.X[random.randint(0, self.N - 1)]
                        X_new = X_best * levy_step + XR + (XR - X_best) * r2
                else:
                    if r1 < 0.5:
                        # X3 – expanded exploitation
                        X_new = (X_best * TF) - (X_M * r1)
                    else:
                        # X4 – narrowed exploitation (Levy)
                        levy_step = levy_flight(size=self.dim)
                        G1 = 2 * (1 - TF)
                        G2 = 2 * TF
                        X_new = X_best - G1 * self.X[i] * r2 - G2 * levy_step

                X_new = self.enforce_bounds(X_new)
                step_size = float(np.linalg.norm(X_new - old_X))

                cfg_new = decode_candidate(X_new)

                if torch.cuda.is_available() and self.device.type == "cuda":
                    torch.cuda.reset_peak_memory_stats(self.device)

                start_t = time.perf_counter()
                f_new = self.fitness(cfg_new)   # this is validation accuracy
                end_t = time.perf_counter()

                self._update_timing_and_gpu_memory(start_t, end_t)
                self.eval_counter += 1

                # MAXIMISATION: replace if f_new > current
                if f_new > self.F[i]:
                    self.X[i] = X_new
                    self.F[i] = f_new

                    if f_new > f_best:
                        f_best = f_new
                        X_best = X_new.copy()

                # update per-candidate personal best (independent of global)
                if f_new > self.candidate_best_val_acc[i]:
                    self.candidate_best_val_acc[i] = f_new
                    self.candidate_best_positions[i] = X_new.copy()

                # update global best
                if f_new > self.global_best_val_acc:
                    self.global_best_val_acc = f_new
                    self.global_best_trial = t
                    self.global_best_candidate_no = self.eval_counter
                    self.global_best_position = X_best.copy()

                # Log this candidate inside this trial — using that candidate's own best
                cand_log = self._make_candidate_log(
                    candidate_no=self.eval_counter,
                    phase=phase,
                    step_size=step_size,
                    cfg=cfg_new,
                    val_acc=f_new,
                    candidate_best_val_acc=self.candidate_best_val_acc[i],
                    candidate_best_position=self.candidate_best_positions[i],
                    TF=TF,
                    r2=r2,
                )
                trial_candidates.append(cand_log)

            self.trials.append(
                {
                    "trial_no": t,
                    "best_val_acc_so_far": float(self.global_best_val_acc),
                    "candidates": trial_candidates,
                }
            )

            print(f"[AO] Iteration {t}/{self.T} | Best val_acc = {f_best:.6f}")

        best_cfg = decode_candidate(X_best)
        return X_best, f_best, best_cfg



#  6. Fitness wrapper for AO

def make_fitness_function(
    train_loader: DataLoader,
    val_loader: DataLoader,
    base_model_name: str,
    num_labels: int,
    id2label: Dict[int, str],
    label2id: Dict[str, int],
    device: torch.device,
    num_epochs: int,
) -> Callable[[Dict[str, Any]], float]:
    """
    Returns a function that AO can call:
        cfg -> validation_accuracy  (we MAXIMISE this).
    """

    def fitness(cfg: Dict[str, Any]) -> float:
        print("  Evaluating config:", cfg)
        val_acc = train_one_model(
            cfg=cfg,
            train_loader=train_loader,
            val_loader=val_loader,
            base_model_name=base_model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
            device=device,
            num_epochs=num_epochs,
        )
        print(f"  -> val_acc (fitness) = {val_acc:.4f}")
        return val_acc

    return fitness

In [ ]:
#  7. Multi-seed main script with JSON result structure

def main():
    # Global config
    seeds = [1,42,999,1234]
    num_seeds = len(seeds)
    total_trials_per_seed = 20      # AO iterations
    num_epochs_per_trial = 3
    candidate_size = 5              # AO population size

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(device)
        torch.cuda.reset_peak_memory_stats(device)
    else:
        gpu_name = "cpu"

    base_model_name = "distilbert-base-uncased"

    print("Loading Emotion dataset once...")
    train_loader, val_loader, test_loader, tokenizer, num_labels, id2label, label2id = load_emotion_dataset(
        model_name=base_model_name,
        max_length=128,
        train_subset=3000,
        batch_size=32,
    )

    # Search space metadata
    search_space = {
        "learning_rate": {
            "type": "log_uniform",
            "min": 1e-6,
            "max": 5e-4,
        },
        "warmup_ratio": {
            "type": "continuous",
            "min": 0.0,
            "max": 0.2,
        },
        "lora_r": {
            "type": "discrete",
            "values": [2, 4, 8, 16, 32],
        },
        "lora_alpha": {
            "type": "discrete",
            "values": [8, 16, 32, 64, 128],
        },
        "lora_dropout": {
            "type": "continuous",
            "min": 0.0,
            "max": 0.3,
        },
        "target_modules": {
            "type": "categorical",
            "values": [
                "attention-only",
                "attention-plus-ffn",
            ],
        },
    }

    seeds_results: List[Dict[str, Any]] = []
    all_val_accs_all_seeds: List[float] = []
    seed_best_accs: List[float] = []
    total_train_time_all_seeds: float = 0.0

    global_best_seed = None
    global_best_val_acc = float("-inf")
    global_best_trial_no = None
    global_best_hyperparams: Dict[str, Any] = {}

    for seed in seeds:
        print("\n=======================================")
        print(f"Running AO for seed {seed}")
        print("=======================================")

        set_seed(seed)

        fitness_fn = make_fitness_function(
            train_loader=train_loader,
            val_loader=val_loader,
            base_model_name=base_model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
            device=device,
            num_epochs=num_epochs_per_trial,
        )

        ao = AquilaOptimizer(
            dim=6,
            fitness_function=fitness_fn,
            device=device,
            pop_size=candidate_size,
            iterations=total_trials_per_seed,
        )

        best_vec, best_fitness, best_cfg = ao.optimize()

        # Collect per-seed stats from ao.trials
        all_candidates_this_seed = [
            cand
            for trial in ao.trials
            for cand in trial["candidates"]
        ]

        if all_candidates_this_seed:
            val_accs = np.array([c["val_acc"] for c in all_candidates_this_seed], dtype=float)
            train_times = np.array([c["train_time_sec"] for c in all_candidates_this_seed], dtype=float)
            gpu_mems = np.array([c["gpu_memory_gb"] for c in all_candidates_this_seed], dtype=float)

            best_idx = int(np.argmax(val_accs))
            best_candidate_no = all_candidates_this_seed[best_idx]["candidate_no"]
            best_accuracy = float(val_accs[best_idx])
            val_acc_mean = float(val_accs.mean())
            val_acc_std = float(val_accs.std(ddof=0))
            total_train_time = float(train_times.sum())
            avg_train_time = float(train_times.mean())
            avg_gpu_memory_gb = float(gpu_mems.mean())
        else:
            best_candidate_no = -1
            best_accuracy = 0.0
            val_acc_mean = 0.0
            val_acc_std = 0.0
            total_train_time = 0.0
            avg_train_time = 0.0
            avg_gpu_memory_gb = 0.0

        all_val_accs_all_seeds.extend([c["val_acc"] for c in all_candidates_this_seed])
        seed_best_accs.append(best_accuracy)
        total_train_time_all_seeds += total_train_time

        if best_accuracy > global_best_val_acc:
            global_best_val_acc = best_accuracy
            global_best_seed = seed
            global_best_trial_no = ao.global_best_trial
            if len(best_cfg["target_modules"]) == 2:
                tm_option = "attention-only"
            else:
                tm_option = "attention-plus-ffn"
            global_best_hyperparams = {
                "learning_rate": float(best_cfg["learning_rate"]),
                "warmup_ratio": float(best_cfg["warmup_ratio"]),
                "lora_r": int(best_cfg["lora_r"]),
                "lora_alpha": int(best_cfg["lora_alpha"]),
                "lora_dropout": float(best_cfg["lora_dropout"]),
                "target_modules_option": tm_option,
            }

        seeds_results.append(
            {
                "seed": seed,
                "trials": ao.trials,
                "best_trial": ao.global_best_trial,
                "best_candidate": ao.global_best_candidate_no,
                "best_accuracy": best_accuracy,
                "val_acc_mean": val_acc_mean,
                "val_acc_std": val_acc_std,
                "avg_train_time_sec": avg_train_time,
                "total_train_time": total_train_time,
                "avg_gpu_memory_gb": avg_gpu_memory_gb,
            }
        )


    if all_val_accs_all_seeds:
        all_val_accs_arr = np.array(all_val_accs_all_seeds, dtype=float)
        val_acc_mean_over_all_trials = float(all_val_accs_arr.mean())
        val_acc_std_over_all_trials = float(all_val_accs_arr.std(ddof=0))
    else:
        val_acc_mean_over_all_trials = 0.0
        val_acc_std_over_all_trials = 0.0

    if seed_best_accs:
        seed_best_arr = np.array(seed_best_accs, dtype=float)
        val_acc_mean_of_seed_bests = float(seed_best_arr.mean())
        val_acc_std_of_seed_bests = float(seed_best_arr.std(ddof=0))
    else:
        val_acc_mean_of_seed_bests = 0.0
        val_acc_std_of_seed_bests = 0.0

    overall_stats = {
        "global_best_seed": global_best_seed,
        "global_best_trial_no": global_best_trial_no,
        "global_best_val_acc": global_best_val_acc,
        "global_best_hyperparams": global_best_hyperparams,
        "val_acc_mean_over_all_trials": val_acc_mean_over_all_trials,
        "val_acc_std_over_all_trials": val_acc_std_over_all_trials,
        "val_acc_mean_of_seed_bests": val_acc_mean_of_seed_bests,
        "val_acc_std_of_seed_bests": val_acc_std_of_seed_bests,
        "total_train_time_sec_over_all_seeds": float(total_train_time_all_seeds),
    }


    #  Final top-level JSON structure
    results = [
        {
            "method": "aquila_optimizer",
            "gpu_name": gpu_name,
            "candidate_size": candidate_size,
            "num_seeds": num_seeds,
            "seeds": seeds,
            "total_trials_per_seed": total_trials_per_seed,
            "num_epochs_per_trial": num_epochs_per_trial,
            "search_space": search_space,
            "seeds_results": seeds_results,
            "overall_stats": overall_stats,
        }
    ]

    print("\n==============================")
    print(" AO MULTI-SEED JSON RESULTS ")
    print("==============================")
    print(json.dumps(results, indent=2))

    out_path = "ao_results_multi_seed.json"
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved AO multi-seed results JSON to: {out_path}")

In [ ]:
if __name__ == "__main__":
    main()


Loading Emotion dataset once...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

...

    Epoch 1/3 - train loss: 1.2331
    Epoch 2/3 - train loss: 0.5465
    Epoch 3/3 - train loss: 0.3482
  -> val_acc (fitness) = 0.8695
  Evaluating config: {'learning_rate': 4.3367253359441845e-05, 'warmup_ratio': 0.09061979612449711, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.0, 'target_modules': ['q_lin', 'v_lin', 'ffn.lin1', 'ffn.lin2']}


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Epoch 1/3 - train loss: 1.6077
    Epoch 2/3 - train loss: 1.2226
    Epoch 3/3 - train loss: 1.0649
  -> val_acc (fitness) = 0.6110
  Evaluating config: {'learning_rate': 6.34281160948043e-05, 'warmup_ratio': 0.03130735674694284, 'lora_r': 16, 'lora_alpha': 64, 'lora_dropout': 0.11508587627748565, 'target_modules': ['q_lin', 'v_lin', 'ffn.lin1', 'ffn.lin2']}


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    Epoch 1/3 - train loss: 1.4559
    Epoch 2/3 - train loss: 1.0277
    Epoch 3/3 - train loss: 0.8655
  -> val_acc (fitness) = 0.6880
[AO] Iteration 20/20 | Best val_acc = 0.910000

 AO MULTI-SEED JSON RESULTS 
[
  {
    "method": "aquila_optimizer",
    "gpu_name": "Tesla T4",
    "candidate_size": 5,
    "num_seeds": 1,
    "seeds": [
      1
    ],
    "total_trials_per_seed": 20,
    "num_epochs_per_trial": 3,
    "search_space": {
      "learning_rate": {
        "type": "log_uniform",
        "min": 1e-06,
        "max": 0.0005
      },
      "warmup_ratio": {
        "type": "continuous",
        "min": 0.0,
        "max": 0.2
      },
      "lora_r": {
        "type": "discrete",
        "values": [
          2,
          4,
          8,
          16,
          32
        ]
      },
      "lora_alpha": {
        "type": "discrete",
        "values": [
          8,
          16,
          32,
          64,
          128
        ]
      },
      "lora_dropout": {
        "type

### Due to four different seeds the upper code has been run 4 time due to GPU time Limit

In [ ]:
import json
import numpy as np
input_files = [
    "ao_results_multi_seed_1.json",
    "ao_results_multi_seed_42.json",
    "ao_results_multi_seed_999.json",
    "ao_results_multi_seed_1234.json",
]

output_file = PROJECT_ROOT / "Results/ao_results_all_seeds_combined.json"


def load_single_seed_result(path):
    with open(path, "r") as f:
        data = json.load(f)
    # Each file is a list with one big dict
    return data[0]


single_seed_runs = [load_single_seed_result(p) for p in input_files]

# -------------------------------------------------------------------
# 2. Sanity checks and basic shared info
# -------------------------------------------------------------------
base_run = single_seed_runs[0]
method = base_run["method"]
gpu_name = base_run.get("gpu_name", None)
candidate_size = base_run["candidate_size"]
total_trials_per_seed = base_run["total_trials_per_seed"]
num_epochs_per_trial = base_run["num_epochs_per_trial"]
search_space = base_run["search_space"]

# Check search_space consistency (optional but good)
for run in single_seed_runs[1:]:
    if run["search_space"] != search_space:
        print("WARNING: search_space differs between files!")

# -------------------------------------------------------------------
# 3. Build combined seeds_results list and collect stats
# -------------------------------------------------------------------
combined_seeds_results = []
all_seed_ids = []
all_val_accs_all_trials = []  # every candidate of every seed
seed_best_accs = []
total_train_time_all_seeds = 0.0

global_best_val_acc = -1.0
global_best_seed = None
global_best_trial_no = None
global_best_hparams = None

for run in single_seed_runs:
    seed_result = run["seeds_results"][0]  # there is only one seed per file
    seed_id = seed_result["seed"]
    all_seed_ids.append(seed_id)
    combined_seeds_results.append(seed_result)

    # accumulate train time
    total_train_time_all_seeds += seed_result["total_train_time"]

    # collect all candidate accs from this seed
    for trial in seed_result["trials"]:
        for cand in trial["candidates"]:
            all_val_accs_all_trials.append(cand["val_acc"])

    # store best for each seed
    seed_best_accs.append(seed_result["best_accuracy"])

    # check global best
    if seed_result["best_accuracy"] > global_best_val_acc:
        global_best_val_acc = seed_result["best_accuracy"]
        global_best_seed = seed_id
        global_best_trial_no = seed_result["best_trial"]
        # find candidate with that accuracy to grab hyperparams
        best_candidate_hparams = None
        for trial in seed_result["trials"]:
            for cand in trial["candidates"]:
                if cand["val_acc"] == seed_result["best_accuracy"]:
                    best_candidate_hparams = cand["hyperparams"]
                    break
            if best_candidate_hparams is not None:
                break

        # map hyperparams into the global schema
        global_best_hparams = {
            "learning_rate": best_candidate_hparams["learning_rate"],
            "warmup_ratio": best_candidate_hparams["warmup_ratio"],
            "lora_r": best_candidate_hparams["lora_r"],
            "lora_alpha": best_candidate_hparams["lora_alpha"],
            "lora_dropout": best_candidate_hparams["lora_dropout"],
            "target_modules_option": best_candidate_hparams["target_modules"],
        }

# convert to numpy for stats
all_val_accs_all_trials = np.array(all_val_accs_all_trials, dtype=float)
seed_best_accs = np.array(seed_best_accs, dtype=float)

val_acc_mean_over_all_trials = float(all_val_accs_all_trials.mean())
val_acc_std_over_all_trials = float(all_val_accs_all_trials.std(ddof=0))

val_acc_mean_of_seed_bests = float(seed_best_accs.mean())
val_acc_std_of_seed_bests = float(seed_best_accs.std(ddof=0))

# -------------------------------------------------------------------
# 4. Build the final combined result object
# -------------------------------------------------------------------
combined_result = {
    "method": method,
    "gpu_name": gpu_name,
    "candidate_size": candidate_size,
    "num_seeds": len(all_seed_ids),
    "seeds": all_seed_ids,
    "total_trials_per_seed": total_trials_per_seed,
    "num_epochs_per_trial": num_epochs_per_trial,
    "search_space": search_space,
    "seeds_results": combined_seeds_results,
    "overall_stats": {
        "global_best_seed": global_best_seed,
        "global_best_trial_no": global_best_trial_no,
        "global_best_val_acc": global_best_val_acc,
        "global_best_hyperparams": global_best_hparams,
        "val_acc_mean_over_all_trials": val_acc_mean_over_all_trials,
        "val_acc_std_over_all_trials": val_acc_std_over_all_trials,
        "val_acc_mean_of_seed_bests": val_acc_mean_of_seed_bests,
        "val_acc_std_of_seed_bests": val_acc_std_of_seed_bests,
        "total_train_time_sec_over_all_seeds": total_train_time_all_seeds,
    },
}

# Top-level list, like your originals
final_json = [combined_result]

with open(output_file, "w") as f:
    json.dump(final_json, f, indent=2)

print(f"Combined results written to: {output_file}")
print("Seeds combined:", all_seed_ids)
print("Global best val_acc:", global_best_val_acc, "from seed", global_best_seed)
